In [4]:
# setup
import re, csv, html, time, requests
from pathlib import Path
from urllib.parse import urljoin, quote

out = Path("data/labins-index")
out.mkdir(parents=True, exist_ok=True)
logf = out / "log.csv"
page = "https://www.labins.org/survey_data/landrecords/glo_volsearch2.cfm?mvol={n:03d}"
hdr = {"User-Agent": "aoa-research (jwrampino@uchicago.edu)"}

# load checkpoint
done = set()
if logf.exists():
    with open(logf) as fh:
        done = {int(r["vol"]) for r in csv.DictReader(fh) if r["pdf_status"] == "200"}
else:
    logf.write_text("vol,page_status,pdf_url,pdf_status,bytes\n")
print(len(done), "volumes already done")

# get with retry on 5xx
def get(url):
    for i in range(3):
        r = requests.get(url, headers=hdr, timeout=60)
        if r.status_code < 500:
            return r
        time.sleep(2 * (i + 1))
    return r

# fetch pages then pdfs
for n in range(1, 279):
    if n in done:
        continue
    p = get(page.format(n=n))
    print(n, "page", p.status_code)
    url, pst, size = "", "", 0
    if p.status_code == 200:
        # find link by anchor text
        m = re.search(r'<a[^>]*href=["\']([^"\']+)["\'][^>]*>\s*All Index and Diagram Pages', p.text, re.I | re.S)
        if m:
            url = quote(urljoin(p.url, html.unescape(m.group(1))), safe=":/&%?=")   # encode spaces
            r = get(url)
            pst = str(r.status_code)
            print(n, "pdf", r.status_code, url)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                (out / f"vol{n:03d}.pdf").write_bytes(r.content)
                size = len(r.content)
            elif r.status_code == 200:
                pst = "not pdf"
        else:
            pst = "no link"
            print(n, "no pdf link")
    with open(logf, "a", newline="") as fh:   # checkpoint per volume
        csv.writer(fh).writerow([n, p.status_code, url, pst, size])
    if p.status_code >= 400 or (pst.isdigit() and int(pst) >= 400):   # stop on any 4xx
        print("stopping at", n)
        break
    time.sleep(2)   # rate limit

# summary
print(len(list(out.glob("vol*.pdf"))), "files saved")

274 volumes already done
83 page 200
83 no pdf link
103 page 200
103 no pdf link
192 page 200
192 no pdf link
217 page 200
217 pdf 404 https://ftp.labins.org/glo_all/Volume217_pdf/Folder%20%201%20Inex%20&%20Diagram_pdf.pdf
stopping at 217
274 files saved


In [5]:
# setup
import csv, time, requests
import pandas as pd
from pathlib import Path
from urllib.parse import quote

out = Path("data/labins-index")
logf = out / "log.csv"
hdr = {"User-Agent": "aoa-research (jwrampino@uchicago.edu)"}

# manual urls
manual = {
    83: "https://ftp.labins.org/glo_all/Volume83_pdf/Folder%20%201%20Index%20&%20Diagarm_pdf.pdf",
    103: "https://ftp.labins.org/glo_all/Volume103_pdf/Folder%20%201%20Index%20&%20Diagame_pdf.pdf"
}

# load log
log = pd.read_csv(logf, dtype={"pdf_url": str, "pdf_status": str}).fillna("")

# fetch manual urls and update log
for n, u in manual.items():
    u = quote(u, safe=":/&%?=")   # encode spaces
    r = requests.get(u, headers=hdr, timeout=60)
    ok = r.status_code == 200 and r.content[:4] == b"%PDF"
    print(n, r.status_code, "pdf" if ok else "not pdf", u)
    if ok:
        (out / f"vol{n:03d}.pdf").write_bytes(r.content)
    row = {"vol": n, "page_status": 200, "pdf_url": u,
           "pdf_status": "200" if ok else str(r.status_code), "bytes": len(r.content) if ok else 0}
    log = log[log["vol"] != n]   # drop old row for this vol
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    time.sleep(1)   # rate limit
log = log.sort_values("vol").reset_index(drop=True)
log.to_csv(logf, index=False)

# remaining missingness
bad = log[(log["pdf_status"] != "200") | (~log["vol"].map(lambda n: (out / f"vol{n:03d}.pdf").exists()))]
gaps = sorted(set(range(1, int(log["vol"].max()) + 1)) - set(log["vol"]))   # vols never logged
print(len(log), "rows,", len(bad), "missing pdf")
print(bad[["vol", "page_status", "pdf_status", "pdf_url"]].to_string(index=False))
print("not in log:", gaps)

83 200 pdf https://ftp.labins.org/glo_all/Volume83_pdf/Folder%20%201%20Index%20&%20Diagarm_pdf.pdf
103 200 pdf https://ftp.labins.org/glo_all/Volume103_pdf/Folder%20%201%20Index%20&%20Diagame_pdf.pdf
278 rows, 2 missing pdf
 vol  page_status pdf_status                                                                                 pdf_url
 192          200    no link                                                                                        
 217          200        404 https://ftp.labins.org/glo_all/Volume217_pdf/Folder%20%201%20Inex%20&%20Diagram_pdf.pdf
not in log: []
